# Generate 48-Case 3D Temperature Surrogate Dataset

Generate a grid-size-invariant surrogate dataset with:

- 4 geometry orientations: 0, 90, 180, 270 degrees
- 3 velocity strengths
- 4 thermal scenarios

Velocity is cached once per geometry orientation and velocity strength, then reused across the four thermal scenarios.

In [ ]:
from pathlib import Path
import json
import sys

PHYSICAL_MODEL_DIR = Path.cwd() / "models" / "physical_model"
VELOCITY_MODEL_DIR = Path.cwd() / "models" / "velocity_calculation"
for path in (PHYSICAL_MODEL_DIR, VELOCITY_MODEL_DIR):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from south_kensington_jupyter import SouthKensingtonConfig
from south_kensington_temperature_3d import Temperature3DScenarioConfig
from generate_temperature_3d_surrogate_dataset import (
    default_geometry_cases,
    default_velocity_cases,
    default_thermal_cases,
    generate_temperature_3d_surrogate_dataset,
)

BUNDLE_ROOT = Path.cwd()
if not (BUNDLE_ROOT / "south_kensington_temperature_3d.py").exists():
    BUNDLE_ROOT = Path("velocity/temperature_field_bundle").resolve()
print("Bundle root:", BUNDLE_ROOT)

## Dataset Configuration


In [ ]:
DATASET_DIR = BUNDLE_ROOT / "outputs" / "temperature_3d_surrogate_dataset_48"
SMOKE_TEST = False
SKIP_COMPLETED = True

geometry_cases = default_geometry_cases()
velocity_cases = default_velocity_cases()
thermal_cases = default_thermal_cases()

if SMOKE_TEST:
    geometry_cases = geometry_cases[:1]
    velocity_cases = velocity_cases[:1]
    thermal_cases = thermal_cases[:1]

print("Dataset dir:", DATASET_DIR)
print("Geometry cases:", geometry_cases)
print("Velocity cases:", velocity_cases)
print("Thermal cases:", thermal_cases)
print("Expected cases:", len(geometry_cases) * len(velocity_cases) * len(thermal_cases))

## Solver Configuration


In [ ]:
base_velocity_config = SouthKensingtonConfig(
    timesteppings=50,
    model_resolution_m=4.0,
    z_dim=10,
    height_scale_m=4.0,
    output_dir=str(DATASET_DIR / "velocity_digit_outputs" / "placeholder"),
)

base_temperature_config = Temperature3DScenarioConfig(
    output_dir=str(DATASET_DIR / "cases" / "placeholder"),
    reuse_velocity_output_dir=str(DATASET_DIR / "velocity_cache" / "placeholder"),
    frame_duration_s=90.0,
    diffusion_coeff_m2_s=1.0,
    velocity_scale=1.0,
    analysis_time_utc="2025-07-25T13:00:00+00:00",
    save_animation=False,
    save_overview_figure=False,
    temperature_solver_backend="auto",
    temperature_solver_device="cuda",
)

print(
    json.dumps(
        {
            "base_velocity_config": base_velocity_config.__dict__,
            "base_temperature_config": base_temperature_config.__dict__,
        },
        indent=2,
    )
)

## Generate Dataset


In [ ]:
summary = generate_temperature_3d_surrogate_dataset(
    DATASET_DIR,
    base_velocity_config=base_velocity_config,
    base_temperature_config=base_temperature_config,
    geometry_cases=geometry_cases,
    velocity_cases=velocity_cases,
    thermal_cases=thermal_cases,
    skip_completed=SKIP_COMPLETED,
)

print(json.dumps(summary, indent=2))

## Inspect Manifest

In [ ]:
manifest_path = DATASET_DIR / "manifest.jsonl"
records = []
if manifest_path.exists():
    with manifest_path.open() as handle:
        records = [json.loads(line) for line in handle if line.strip()]
print("Manifest records:", len(records))
if records:
    print(json.dumps(records[0], indent=2))
    print(json.dumps(records[-1], indent=2))

## Dataset Visualization


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

manifest_path = DATASET_DIR / "manifest.jsonl"
records = []
if manifest_path.exists():
    with manifest_path.open() as handle:
        records = [json.loads(line) for line in handle if line.strip()]

print("Completed records:", len(records))
if not records:
    raise RuntimeError("No generated cases found yet. Run the generation cell first.")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, key, title in zip(
    axes,
    ["geometry_case_id", "velocity_case_id", "thermal_case_id"],
    ["Geometry orientation", "Velocity strength", "Thermal scenario"],
):
    counts = Counter(record[key] for record in records)
    labels = sorted(counts)
    ax.bar(labels, [counts[label] for label in labels], color="#4C78A8")
    ax.set_title(title)
    ax.set_ylabel("cases")
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Compare actual temperature fields across the four geometry/wind-direction cases.
# Preference: medium velocity (v01) + baseline thermal case (t01).
records_by_geometry = {}
for geometry_id in sorted({record["geometry_case_id"] for record in records}):
    candidates = [record for record in records if record["geometry_case_id"] == geometry_id]
    preferred = [
        record
        for record in candidates
        if record["velocity_case_id"] == "v01" and record["thermal_case_id"] == "t01"
    ]
    records_by_geometry[geometry_id] = preferred[0] if preferred else candidates[0]

selected = [records_by_geometry[key] for key in sorted(records_by_geometry)]
arrays = []
metadata = []
for record in selected:
    case_dir = Path(record["case_dir"])
    temp = np.load(case_dir / "temperature_fields_3d_masked_c.npy")
    velocity_cache_dir = Path(record["velocity_cache_dir"])
    building_mask = np.load(velocity_cache_dir / "building_mask_2d.npy").astype(bool)
    frame_id = temp.shape[0] - 1
    z = temp.shape[1] // 8
    arrays.append(temp[frame_id, z])
    metadata.append((record, building_mask, frame_id, z))

vmin = min(float(np.nanpercentile(arr, 2)) for arr in arrays)
vmax = max(float(np.nanpercentile(arr, 98)) for arr in arrays)
fig, axes = plt.subplots(1, len(arrays), figsize=(4.7 * len(arrays), 4.4), constrained_layout=True)
if len(arrays) == 1:
    axes = [axes]
for ax, arr, (record, building_mask, frame_id, z) in zip(axes, arrays, metadata):
    im = ax.imshow(arr, cmap="YlOrRd", vmin=vmin, vmax=vmax, origin="upper")
    ax.contour(
        building_mask.astype(float), levels=[0.5], colors="black", linewidths=0.45, alpha=0.75
    )
    ax.set_title(
        f"{record['geometry_case_id']} ({record['geometry_rotation_deg']} deg)\n"
        f"{record['velocity_case_id']} | {record['thermal_case_id']} | frame {frame_id} | z={z}"
    )
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.78, label="deg C")
plt.show()

In [ ]:
# Inspect time evolution for one representative case with building geometry overlaid.
sample_record = records_by_geometry.get("g000", selected[0])
case_dir = Path(sample_record["case_dir"])
velocity_cache_dir = Path(sample_record["velocity_cache_dir"])
temp = np.load(case_dir / "temperature_fields_3d_masked_c.npy")
building_mask = np.load(velocity_cache_dir / "building_mask_2d.npy").astype(bool)
safe_z = temp.shape[1] // 16
frame_ids = [1, temp.shape[0] // 2, temp.shape[0] - 1]

vmin = float(np.nanpercentile(temp[:, safe_z], 2))
vmax = float(np.nanpercentile(temp[:, safe_z], 98))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
for ax, frame_id in zip(axes, frame_ids):
    im = ax.imshow(temp[frame_id, safe_z], cmap="YlOrRd", vmin=vmin, vmax=vmax, origin="upper")
    ax.contour(
        building_mask.astype(float), levels=[0.5], colors="black", linewidths=0.45, alpha=0.75
    )
    ax.set_title(f"{sample_record['case_id']}\nframe {frame_id} | z={safe_z}")
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.78, label="deg C")
plt.show()